# imports

In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.inspection import permutation_importance


# load model

In [4]:
from  keras.models import load_model

model = load_model('model.keras')

# add model to mlflow

In [8]:
import mlflow
import os
import tensorflow as tf

mlflow.set_tracking_uri("https://mlflow.srv1197470.hstgr.cloud/")
os.environ['MLFLOW_TRACKING_USERNAME'] = "admin"
os.environ['MLFLOW_TRACKING_PASSWORD'] = "password1234"

mlflow.set_experiment(experiment_name='Art-tuto')

with mlflow.start_run(run_name='Vision_model'):
    mlflow.keras.log_model(model,"vision model")

2026/02/27 15:43:31 INFO mlflow.tracking.fluent: Experiment with name 'Art-tuto' does not exist. Creating a new experiment.
2026/02/27 15:43:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/27 15:43:33 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/02/27 15:44:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Vision_model at: https://mlflow.srv1197470.hstgr.cloud/#/experiments/1/runs/8f565641085e4b20bdf81a06d85c465e
🧪 View experiment at: https://mlflow.srv1197470.hstgr.cloud/#/experiments/1


# load test data

In [ ]:
data_dir = './PetImages_ol' 
img_size = (224, 224) # resize images so they are compatible with most pretained image models
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.25,
    subset="training",
    seed=123,  #makes this split repeatable, so data is split the same way every tume. like random_state=42
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.25,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

Found 24998 files belonging to 2 classes.
Using 18749 files for training.
Found 24998 files belonging to 2 classes.
Using 6249 files for validation.


# load ART tools

## classifiers

In [ ]:
from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import KerasClassifier
import numpy as np


classifier = KerasClassifier(model=model, use_logits=False, clip_values=(0.0, 1.0), preprocessing_defences=None)

x = np.random.rand(1, 224, 224, 3).astype("float32")

attack = FastGradientMethod(estimator=classifier, eps=0.01)

clean_correct = 0
adv_correct = 0
total = 0



Clean accuracy: 0.9799967994879181
Adversarial accuracy: 0.9801568250920147
Attack success rate: 0.01984317490798526


## Testing

In [ ]:
for x_batch, y_batch in val_ds:
    # Generate adversarial examples
    x_np = x_batch.numpy()
    y_np = y_batch.numpy()
    
    # Clean predictions
    preds_clean = classifier.predict(x_np)
    preds_clean_labels = (preds_clean > 0.5).astype(int).flatten()
    clean_correct += (preds_clean_labels == y_np.flatten()).sum()


    x_adv = attack.generate(x_np)

    # Adversarial predictions
    preds_adv = classifier.predict(x_adv)
    preds_adv_labels = (preds_adv > 0.5).astype(int).flatten()
    adv_correct += (preds_adv_labels == y_np.flatten()).sum()

    total += x_np.shape[0]

clean_accuracy = clean_correct / total
adv_accuracy = adv_correct / total
attack_success_rate = 1 - adv_accuracy

print("Clean accuracy:", clean_accuracy)
print("Adversarial accuracy:", adv_accuracy)
print("Attack success rate:", attack_success_rate)

